In [1]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

FILE="Water Quality.csv"
DATE="ActivityStartDate"
PARAM="CharacteristicName"
VALUE="ResultValue"
OUT="task3_trend_outputs"
os.makedirs(OUT,exist_ok=True)
df=pd.read_csv(FILE,low_memory=False)
required=[DATE,PARAM,VALUE]
missing=[c for c in required if c not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}; available: {list(df.columns)}")
df[DATE]=pd.to_datetime(df[DATE],errors="coerce")
df[VALUE]=pd.to_numeric(df[VALUE],errors="coerce")
df=df.dropna(subset=required)
if df.empty: raise ValueError("No valid dates and numeric measurements.")

def safe(s): return re.sub(r"[^A-Za-z0-9_-]+","_",str(s))[:70]
params=df[PARAM].value_counts().head(10).index
rows=[]
for param in params:
    series=df[df[PARAM]==param].set_index(DATE)[VALUE].sort_index().resample("MS").median()
    if series.notna().sum()<3: continue
    filled=series.interpolate(method="time",limit_direction="both")
    r3=filled.rolling(3,min_periods=1).mean()
    r6=filled.rolling(6,min_periods=1).mean()
    r12=filled.rolling(12,min_periods=1).mean()
    valid=filled.dropna()
    x=np.arange(len(valid),dtype=float)
    slope,intercept,r,p,stderr=linregress(x,valid.values)
    line=intercept+slope*x
    name=safe(param)
    pd.DataFrame({"monthly_median":series,"filled_for_plot":filled,
        "rolling_3_month":r3,"rolling_6_month":r6,"rolling_12_month":r12}).to_csv(f"{OUT}/{name}_series.csv",index_label="Date")
    plt.figure(figsize=(12,5))
    plt.plot(series.index,series.values,alpha=.45,marker=".",label="Monthly median")
    plt.plot(filled.index,r3,label="3-month average")
    plt.plot(filled.index,r6,label="6-month average")
    plt.plot(filled.index,r12,label="12-month average")
    plt.plot(valid.index,line,"--",linewidth=2,label="Linear trend")
    plt.title(f"Long-term trend: {param}"); plt.xlabel("Date"); plt.ylabel("ResultValue")
    plt.legend(); plt.tight_layout(); plt.savefig(f"{OUT}/{name}_trend.png",dpi=220); plt.close()
    rows.append({"Parameter":param,"Monthly_points":len(valid),"Slope_per_month":slope,
        "Approx_slope_per_year":slope*12,"R_squared":r*r,"Trend_p_value":p,
        "Trend_direction":"Increasing" if slope>0 else "Decreasing" if slope<0 else "Flat"})
result=pd.DataFrame(rows)
result.to_csv(f"{OUT}/long_term_trend_summary.csv",index=False)
print(f"Done. Outputs: {OUT}/")
if not result.empty: print(result.round(5).to_string(index=False))
print("Moving averages smooth short-term variation. The linear slope estimates direction over the observed period, not causation. Missing monthly values are interpolated for plotting; disclose this in your report.")


Done. Outputs: task3_trend_outputs/
                  Parameter  Monthly_points  Slope_per_month  Approx_slope_per_year  R_squared  Trend_p_value Trend_direction
       Specific conductance             288          0.12469                1.49623    0.10768        0.00000      Increasing
                         pH             288          0.00475                0.05698    0.38089        0.00000      Increasing
Total Nitrogen, mixed forms             287          0.00116                0.01391    0.44488        0.00000      Increasing
                  Magnesium             287          0.00051                0.00607    0.03013        0.00317      Increasing
                    Calcium             287          0.00595                0.07138    0.10013        0.00000      Increasing
                     Sodium             287          0.00441                0.05296    0.07184        0.00000      Increasing
             Organic carbon             287          0.00091                0.0109